# Data Cleaning and Preprocessing
## Sri Lanka Rice Production Forecasting Project

**Objective:** This notebook consolidates raw rice production data from multiple CSV files (Maha and Yala seasons, 2004-2023), performs data cleaning, and prepares the dataset for analysis and forecasting.

**Author:** Visura Rodrigo  
**Last Updated:** January 26, 2026

---

### Table of Contents
1. [Data Loading](#data-loading)
2. [Feature Engineering](#feature-engineering)
3. [Data Cleaning](#data-cleaning)
4. [Data Export](#data-export)

## 1. Data Loading <a id='data-loading'></a>

This section loads all CSV files from the `data/raw` directory. Each file represents rice production data for a specific season (Maha or Yala) and year. The files are consolidated into a single DataFrame for further processing.

In [1]:
# Import required libraries
import pandas as pd
import os
import glob

# Define path to raw data directory
folder_path = os.path.join('..', 'data', 'raw')

# Retrieve all CSV files from the directory
all_files = glob.glob(os.path.join(folder_path, "*.csv"))
print(f"✅ Found {len(all_files)} CSV files in {folder_path}")

# Initialize list to store individual DataFrames
df_list = []

# Load and process each CSV file
for filename in all_files:
    try:
        # Read CSV file
        temp_df = pd.read_csv(filename)
        
        # Add source filename as a column for traceability
        # This enables extraction of year and season information later
        temp_df['Source_File'] = os.path.basename(filename)
        
        # Append to list
        df_list.append(temp_df)
        
    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")

# Concatenate all DataFrames into a single master DataFrame
if df_list:
    full_df = pd.concat(df_list, ignore_index=True)
    print(f"\n✅ Successfully consolidated all files!")
    print(f"   Total Rows: {full_df.shape[0]:,}")
    print(f"   Total Columns: {full_df.shape[1]}")
else:
    print("❌ No dataframes to merge.")

# Display sample records
print("\n--- First 5 Records ---")
display(full_df.head())
print("\n--- Last 5 Records ---")
display(full_df.tail())

✅ Found 38 files.

✅ Successfully merged all files!
Total Rows: 1039
Total Columns: 16


,District,Major_Schemes_Sown,Minor_Schemes_Sown,Rainfed_Sown,All_Schemes_Sown,Major_Schemes_Harvested,Minor_Schemes_Harvested,Rainfed_Harvested,All_Schemes_Harvested,Major_Schemes_Yield,Minor_Schemes_Yield,Rainfed_Yield,Average_Yield,Nett_Extent_Harvested,Total_Production,Source_File
0,COLOMBO,1141,85,3367,4594,85,1141,3365,4592,2564,3426,3218,3251,3903,12687,2004 - 2005 Maha.csv
1,GAMPAHA,1170,1192,7811,10173,1191,1170,7809,10170,3226,3500,3430,3408,8644,29460,2004 - 2005 Maha.csv
2,KALUTARA,1938,96,11608,13642,93,1936,11581,13610,3344,2594,2913,2865,12556,35971,2004 - 2005 Maha.csv
3,GALLE,175,-,14323,14498,-,174,13769,13943,-,3614,3719,3711,10912,40494,2004 - 2005 Maha.csv
4,MATARA,2809,3615,8439,14862,3610,2752,8333,14695,4415,3859,3498,3785,11602,43912,2004 - 2005 Maha.csv


,District,Major_Schemes_Sown,Minor_Schemes_Sown,Rainfed_Sown,All_Schemes_Sown,Major_Schemes_Harvested,Minor_Schemes_Harvested,Rainfed_Harvested,All_Schemes_Harvested,Major_Schemes_Yield,Minor_Schemes_Yield,Rainfed_Yield,Average_Yield,Nett_Extent_Harvested,Total_Production,Source_File
1034,BADULLA,8251,4785,163,13199,8251,4762,163,13176,4592,4491,4451,4554,11200,51003,2023 Yala.csv
1035,MONARAGALA,8277,6787,1497,16561,8274,6737,1479,16490,4455,3823,2555,4026,16160,65066,2023 Yala.csv
1036,RATNAPURA,4430,4584,1473,10487,4397,4564,1463,10424,5661,3164,3002,4195,8861,37169,2023 Yala.csv
1037,KEGALLE,-,690,5804,6494,-,690,5692,6382,-,3142,2954,2974,5979,17782,2023 Yala.csv
1038,SRI_LANKA,320176,121856,63044,505076,316772,117005,57895,491672,4223,3533,3111,3822,440304,1817391,2023 Yala.csv


## 2. Feature Engineering <a id='feature-engineering'></a>

Extract temporal features (Season and Year) from the source filename. These features are critical for time-series analysis and forecasting models.

In [2]:
# Define function to extract season from filename
def get_season(filename):
    """
    Extracts the cultivation season from the filename.
    
    Args:
        filename (str): Source filename
        
    Returns:
        str: 'Maha' or 'Yala' season, or 'Unknown' if not found
    """
    filename_lower = filename.lower()
    if 'maha' in filename_lower:
        return 'Maha'
    elif 'yala' in filename_lower:
        return 'Yala'
    return 'Unknown'

# Define function to extract year from filename
def get_year(filename):
    """
    Extracts the year/period from the filename.
    
    Args:
        filename (str): Source filename
        
    Returns:
        str: Year or year range (e.g., '2005' or '2004 - 2005')
    """
    # Remove file extension and season keywords
    clean_name = filename.lower().replace('.csv', '').replace('maha', '').replace('yala', '')
    return clean_name.strip()

# Apply feature extraction
full_df['Season'] = full_df['Source_File'].apply(get_season)
full_df['Year'] = full_df['Source_File'].apply(get_year)

# Validate column structure
print("=== Dataset Structure ===")
print(f"Total Columns: {len(full_df.columns)}")
print(f"\nColumn Names:")
print(full_df.columns.tolist())

# Display sample of extracted features
print("\n=== Sample of Extracted Features ===")
display(full_df[['Source_File', 'Year', 'Season']].head(10))

--- Current Column Names ---
['District', 'Major_Schemes_Sown', 'Minor_Schemes_Sown', 'Rainfed_Sown', 'All_Schemes_Sown', 'Major_Schemes_Harvested', 'Minor_Schemes_Harvested', 'Rainfed_Harvested', 'All_Schemes_Harvested', 'Major_Schemes_Yield', 'Minor_Schemes_Yield', 'Rainfed_Yield', 'Average_Yield', 'Nett_Extent_Harvested', 'Total_Production', 'Source_File', 'Season', 'Year']


,Source_File,Year,Season
0,2004 - 2005 Maha.csv,2004 - 2005,Maha
1,2004 - 2005 Maha.csv,2004 - 2005,Maha
2,2004 - 2005 Maha.csv,2004 - 2005,Maha
3,2004 - 2005 Maha.csv,2004 - 2005,Maha
4,2004 - 2005 Maha.csv,2004 - 2005,Maha


## 3. Data Cleaning <a id='data-cleaning'></a>

This section performs comprehensive data cleaning operations:
- **Numeric Formatting:** Remove special characters (commas, hyphens, underscores) from numeric columns
- **Type Conversion:** Convert numeric columns to appropriate data types
- **Data Validation:** Identify and handle missing/invalid values
- **Row Filtering:** Remove aggregate summary rows (e.g., 'Total', 'Sri Lanka')

In [4]:
# Identify numeric columns (exclude categorical/text columns)
numeric_cols = [col for col in full_df.columns 
                if col not in ['District', 'Source_File', 'Season', 'Year']]

# Define function to clean and standardize numeric values
def clean_currency(value):
    """
    Cleans numeric values by removing formatting characters.
    
    Args:
        value: Input value (may be string or numeric)
        
    Returns:
        float: Cleaned numeric value, or 0 if conversion fails
    """
    if isinstance(value, str):
        # Remove common formatting characters
        clean_str = value.replace(',', '').replace('-', '').replace('_', '')
        
        # Handle empty strings
        if clean_str.strip() == '':
            return 0
        
        # Attempt conversion to float
        try:
            return float(clean_str)
        except ValueError:
            return 0
    
    return value

# Apply cleaning function to all numeric columns
print("=== Cleaning Numeric Columns ===")
for col in numeric_cols:
    full_df[col] = full_df[col].apply(clean_currency)
    full_df[col] = pd.to_numeric(full_df[col], errors='coerce').fillna(0)

print(f"✅ Successfully cleaned {len(numeric_cols)} numeric columns")

# Analyze district names for data quality issues
print("\n=== District Analysis ===")
unique_districts = sorted(full_df['District'].unique().astype(str))
print(f"Total Unique Districts: {len(unique_districts)}")
print(f"\nDistrict List:")
for district in unique_districts:
    print(f"  • {district}")

# Remove aggregate summary rows
print("\n=== Removing Summary Rows ===")
rows_before = len(full_df)

# Filter out 'Total' and 'Sri Lanka' aggregate rows
full_df = full_df[~full_df['District'].astype(str).str.contains('Total', case=False, na=False)]
full_df = full_df[~full_df['District'].astype(str).str.contains('Sri Lanka', case=False, na=False)]
full_df = full_df[~full_df['District'].astype(str).str.contains('final', case=False, na=False)]

rows_after = len(full_df)
removed_rows = rows_before - rows_after

print(f"✅ Removed {removed_rows} aggregate summary rows")
print(f"   Remaining Records: {rows_after:,}")

✅ Numeric columns cleaned!

--- Unique Districts ---
['AMPARA', 'AMPARA*', 'ANURADHAPURA', 'BADULLA', 'BATTICALOA', 'COLOMBO', 'GALLE', 'GAMPAHA', 'HAMBANTOTA', 'JAFFNA', 'KALUTARA', 'KANDY', 'KEGALLE', 'KILLINOCHCHI', 'KILLINOCHCHI**', 'KURUNEGALA', 'MAHAWELI', 'MAHAWELI_H', 'MANNAR', 'MANNAR**', 'MATALE', 'MATARA', 'MONARAGALA', 'MULATIVU', 'MULATIVU**', 'NUWARAELIYA', 'POLONNARUWA', 'POLONNARUWA*', 'PUTTALAM', 'RATNAPURA', 'SRI_LANKA', 'TRINCOMALEE', 'Total', 'UDAWALAWE', 'UDA_WALAWE', 'VAVUNIYA', 'final']

Removed 1 summary rows.


### Data Quality Summary

In [6]:
# Generate comprehensive data quality report
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

print(f"\n📊 Dataset Dimensions:")
print(f"   Rows: {full_df.shape[0]:,}")
print(f"   Columns: {full_df.shape[1]}")

print(f"\n📅 Temporal Coverage:")
print(f"   Seasons: {full_df['Season'].unique().tolist()}")
print(f"   Year Range: {full_df['Year'].nunique()} unique periods")

print(f"\n🗺️  Geographic Coverage:")
print(f"   Districts: {full_df['District'].nunique()} unique districts")

print(f"\n❓ Missing Values:")
missing_summary = full_df.isnull().sum()
missing_cols = missing_summary[missing_summary > 0]
if len(missing_cols) > 0:
    for col, count in missing_cols.items():
        print(f"   {col}: {count} ({count/len(full_df)*100:.2f}%)")
else:
    print("   ✅ No missing values detected")

print(f"\n💾 Memory Usage:")
print(f"   {full_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "=" * 60)

DATA QUALITY REPORT

📊 Dataset Dimensions:
   Rows: 1,038
   Columns: 18

📅 Temporal Coverage:
   Seasons: ['Maha', 'Yala']
   Year Range: 38 unique periods

🗺️  Geographic Coverage:
   Districts: 36 unique districts

❓ Missing Values:
   ✅ No missing values detected

💾 Memory Usage:
   0.35 MB



## 4. Data Export <a id='data-export'></a>

Export the cleaned and processed dataset to the `data/processed` directory for use in subsequent analysis and modeling phases.

In [5]:
# Define output path for cleaned dataset
output_path = os.path.join('..', 'data', 'processed', 'paddy_data_cleaned.csv')

# Ensure output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Export cleaned dataset to CSV
full_df.to_csv(output_path, index=False)

print("=" * 60)
print("EXPORT SUCCESSFUL")
print("=" * 60)
print(f"\n📁 File Location: {output_path}")
print(f"📊 Records Exported: {len(full_df):,}")
print(f"📋 Columns Exported: {len(full_df.columns)}")
print(f"\n✅ Cleaned dataset is ready for analysis and modeling!")
print("=" * 60)

✅ Clean data saved to: ..\data\processed\paddy_data_cleaned.csv


---

## Summary

This notebook successfully completed the following data preparation tasks:

✅ **Data Consolidation:** Merged 38 CSV files spanning 2004-2023  
✅ **Feature Engineering:** Extracted Season and Year from filenames  
✅ **Data Cleaning:** Standardized numeric formats and removed invalid characters  
✅ **Quality Assurance:** Filtered aggregate rows and validated data integrity  
✅ **Data Export:** Saved cleaned dataset to `data/processed/paddy_data_cleaned.csv`

### Next Steps

1. **Exploratory Data Analysis (EDA):** Analyze trends, patterns, and seasonality
2. **Feature Engineering:** Create additional predictive features
3. **Model Development:** Build and train forecasting models
4. **Model Evaluation:** Assess model performance and accuracy

---

**End of Notebook**